# MRI processing

In [1]:
# import libraries
import numpy as np 
import pandas as pd 
import os, fnmatch
import nibabel as nib
import matplotlib.pyplot as plt
from scipy.stats import iqr

from joblib import Parallel, delayed
import logging

In [34]:
pd.set_option('display.max_rows', 20)
pd.set_option('display.max_columns', 30)

In [35]:
# upload metafile
#meta2 = pd.read_csv('metafile_completing/metafile_completed_ADNI_A4_processed_02_06_2024.csv') #shuffled on 10_04_2025
#meta2= meta2.sample(frac=1, ignore_index = True)
#meta2.to_csv('metafile_completing/metafile_completed_ADNI_A4_processed_02_06_2024_shuffled.csv')

meta2 = pd.read_csv('metafile_completing/metafile_completed_ADNI_A4_processed_02_06_2024_shuffled.csv')


In [36]:
meta2['Modality'].value_counts()

MRI    42804
PET    22476
Name: Modality, dtype: int64

In [37]:
# selecting all MRI and only amyloid PET scans
pd.set_option('display.max_rows', 20)
meta3 = meta2[(meta2.Modality == 'MRI') | 
      (meta2.modality_subtype == 'AV45')]#|
#     (meta2.modality_subtype == 'FBB')|
#     (meta2.modality_subtype == 'PIB')]

In [38]:
# selecting only T1 and amyloid PET scans with needed pre-proccesing 
'''meta4 = meta3[((meta3.Project == 'ADNI')&((meta3.Description.str.contains('MP*RAGE|T1|IR*SPGR')) | 
     (meta3.Description.str.contains('Co-registered, Averaged'))))|
           (( meta3.Project == 'A4')&((meta3.modality_subtype.str.contains('T1')) | 
     (meta3.Description.str.contains('Flor*'))))]'''

"meta4 = meta3[((meta3.Project == 'ADNI')&((meta3.Description.str.contains('MP*RAGE|T1|IR*SPGR')) | \n     (meta3.Description.str.contains('Co-registered, Averaged'))))|\n           (( meta3.Project == 'A4')&((meta3.modality_subtype.str.contains('T1')) | \n     (meta3.Description.str.contains('Flor*'))))]"

In [39]:
pet_mask = ((meta3.Modality == 'PET')&(meta3.Description.str.contains('Co-registered, Averaged')|
            meta3.Description.str.contains('^Flor.*')))
mri_mask = (meta3.Modality == 'MRI')&(meta3.Description.str.contains('MP.*RAGE|T1|IR.*SPGR')|
            meta3.modality_subtype.str.contains('T1'))

In [40]:
meta4 = meta3[pet_mask|mri_mask]
meta4.reset_index(drop=True, inplace = True)

In [41]:
meta4[meta4.Project == 'A4'].modality_subtype.value_counts()

AV45    4486
T1      1878
Name: modality_subtype, dtype: int64

In [42]:
#pd.set_option('display.max_rows', None)
meta4[meta4.Project == 'ADNI'].modality_subtype.value_counts()

MPRAGE                         10738
MPRAGE GRAPPA2                  4104
MP-RAGE                         3600
AV45                            3004
MPRAGE Repeat                   2193
                               ...  
SAG 3D MPRAGE, no angle            1
Accelerated Satittal MPRAGE        1
MPRAGE_repeat                      1
REPEAT MP-RAGE                     1
MPRAGE-REPEAT                      1
Name: modality_subtype, Length: 171, dtype: int64

In [43]:
meta4 = meta4.copy()
meta4['year'] = ''
for i in range(meta4.shape[0]):
    meta4['year'].values[i] = meta4['Study.Date'].values[i].split('/')[2]

In [44]:
meta4['Modality'].value_counts()

MRI    37003
PET     7490
Name: Modality, dtype: int64

In [45]:
pet_meta = meta4[meta4.Modality == 'PET']
pet_meta.reset_index(drop=True, inplace = True)

mri_meta = meta4[meta4.Modality == 'MRI']
mri_meta.reset_index(drop=True, inplace = True)



In [46]:
pet_meta.columns

Index(['Unnamed: 0', 'Unnamed: 0.1', 'Subject.ID', 'Project', 'Phase', 'Sex',
       'Weight', 'Research.Group', 'Visit', 'Study.Date', 'Archive.Date',
       'Age', 'Modality', 'Description', 'Type', 'Imaging.Protocol',
       'Image.ID', 'VISCODE', 'Image.Data.ID', 'modality_subtype', 'PATH',
       'year'],
      dtype='object')

In [47]:

pet_mri_tab = pet_meta[['Subject.ID','Project','Phase' ,'Sex','Weight','Research.Group','VISCODE','Study.Date','Age','Modality','Description','Imaging.Protocol','Image.Data.ID','modality_subtype','PATH']]
pet_mri_tab = pet_mri_tab.copy()
pet_mri_tab['MRI_subID']= ''
pet_mri_tab['MRI_Modality']= ''
pet_mri_tab['MRI_Research.Group']= ''
pet_mri_tab['MRI_Age']= np.nan
pet_mri_tab['MRI_VISCODE']= ''
pet_mri_tab['MRI_studydate']= ''
pet_mri_tab['MRI_Research.Group']= ''
pet_mri_tab['MRI_Description']= ''
pet_mri_tab['MRI_Imaging.protocol']= ''
pet_mri_tab['MRI_Type']= ''
pet_mri_tab['MRI_ID']= ''
pet_mri_tab['MRI_PATH']= ''

In [48]:
pet_meta.shape[0]

7490

In [49]:
pet_meta['Phase'].value_counts()

ADNI 2     1886
ADNI 3      863
ADNI GO     255
Name: Phase, dtype: int64

In [50]:
pet_meta['VISCODE'].value_counts()

SCV2    4486
v03      754
v21      521
bl       362
init     313
y2       263
v41      251
v11      160
v31      113
m48       87
v06       67
y4        49
m60       37
v51       20
m36        7
Name: VISCODE, dtype: int64

In [51]:
# let's try again 

suc = 0 
er = 0

#for i in range(1992):
for i in range(pet_meta.shape[0]):
    pet_id = pet_meta.iloc[i,:]
    
    mri_data = mri_meta.copy()
    
    mri_data= mri_data[mri_data['Subject.ID']==pet_id['Subject.ID']]
                        
    mri_data['age_diff'] = abs(float(pet_id.Age) - mri_data.Age.astype(float))
    
    mri_data= mri_data[((mri_data['Age'] == pet_id['Age'])
                        | (mri_data['VISCODE'] == pet_id['VISCODE'])
                        |(abs(mri_data.age_diff)<=1))]
                          
    if mri_data.shape[0]== 0:
        er += 1
        del mri_data
        print(str(i) + ' '+ pet_id['Subject.ID']+' error')
        continue
        
    ideal_mri = mri_data[mri_data['VISCODE'] == pet_id['VISCODE']]
    if ideal_mri.shape[0] == 0:
        ideal_mri = mri_data[mri_data['Age'] == pet_id['Age']]
    if ideal_mri.shape[0] == 0:
        ideal_mri = mri_data[mri_data.age_diff == mri_data.age_diff.min()]
    if ideal_mri.shape[0] == 0:
        er += 1
        del mri_data
        print(str(i) + ' '+ pet_id['Subject.ID']+' error')
        continue
            
    mri_data_orig = ideal_mri[ideal_mri['Type'] == 'Original']
    mri_data_proc = ideal_mri[ideal_mri['Type'] == 'Processed']
    
    if pet_id['Phase'] == 'ADNI 3' and mri_data_orig.shape[0]>0:
        suc += 1
        pet_mri_tab['MRI_subID'].values[i]= mri_data_orig.iloc[0,:]['Subject.ID']
        pet_mri_tab['MRI_Modality'].values[i]= mri_data_orig.iloc[0,:]['Modality']
        pet_mri_tab['MRI_Research.Group'].values[i]= mri_data_orig.iloc[0,:]['Research.Group']
        pet_mri_tab['MRI_Age'].values[i]= mri_data_orig.iloc[0,:]['Age']
        pet_mri_tab['MRI_VISCODE'].values[i]= mri_data_orig.iloc[0,:]['VISCODE']
        pet_mri_tab['MRI_studydate'].values[i]= mri_data_orig.iloc[0,:]['Study.Date']
        pet_mri_tab['MRI_Description'].values[i]= mri_data_orig.iloc[0,:]['Description']
        pet_mri_tab['MRI_Imaging.protocol'].values[i]= mri_data_orig.iloc[0,:]['Imaging.Protocol']
        pet_mri_tab['MRI_Type'].values[i]= mri_data_orig.iloc[0,:]['Type']
        pet_mri_tab['MRI_ID'].values[i]= mri_data_orig.iloc[0,:]['Image.Data.ID']
        pet_mri_tab['MRI_PATH'].values[i]= mri_data_orig.iloc[0,:]['PATH']

    elif ((pet_id['Phase'] != 'ADNI 3' and mri_data_proc.shape[0]>0) or 
          (pet_id['Project'] == 'A4' and mri_data_proc.shape[0]>0)):
        suc += 1
        pet_mri_tab['MRI_subID'].values[i]= mri_data_proc.iloc[0,:]['Subject.ID']
        pet_mri_tab['MRI_Modality'].values[i]= mri_data_proc.iloc[0,:]['Modality']
        pet_mri_tab['MRI_Research.Group'].values[i]= mri_data_proc.iloc[0,:]['Research.Group']
        pet_mri_tab['MRI_Age'].values[i]= mri_data_proc.iloc[0,:]['Age']
        pet_mri_tab['MRI_VISCODE'].values[i]= mri_data_proc.iloc[0,:]['VISCODE']
        pet_mri_tab['MRI_studydate'].values[i]= mri_data_proc.iloc[0,:]['Study.Date']
        pet_mri_tab['MRI_Description'].values[i]= mri_data_proc.iloc[0,:]['Description']
        pet_mri_tab['MRI_Imaging.protocol'].values[i]= mri_data_proc.iloc[0,:]['Imaging.Protocol']
        pet_mri_tab['MRI_Type'].values[i]= mri_data_proc.iloc[0,:]['Type']
        pet_mri_tab['MRI_ID'].values[i]= mri_data_proc.iloc[0,:]['Image.Data.ID']
        pet_mri_tab['MRI_PATH'].values[i]= mri_data_proc.iloc[0,:]['PATH']


0 B61659759 error
3 B49983219 error
7 B87425311 error
10 B90196559 error
11 B47761872 error
15 B75038203 error
16 B60794280 error
32 B18601506 error
34 B38322283 error
37 B77307945 error
41 B84642422 error
42 B55412026 error
43 B67551148 error
47 B63119332 error
52 B28071973 error
53 B48916189 error
54 B93878024 error
56 B48995724 error
60 B75984003 error
61 B16386965 error
62 B41388843 error
63 B92918238 error
65 B18581535 error
67 B75057616 error
70 B15283443 error
72 B86955049 error
75 B78518223 error
76 B68756730 error
77 B31427401 error
79 B15099158 error
84 B44674253 error
86 B68104601 error
90 B71778346 error
91 B60427965 error
93 B75863045 error
96 B97669016 error
100 B55663696 error
101 B70058389 error
104 B72085567 error
108 B72869324 error
111 B58827256 error
117 B43590060 error
119 B46644870 error
120 B72580739 error
121 B32093768 error
125 B22846057 error
126 B99577525 error
127 B99367143 error
131 B67984752 error
132 B43447426 error
136 B97948712 error
141 B43634812 error

1047 B18100327 error
1050 B79652440 error
1052 B50913849 error
1058 B91764226 error
1059 B96597815 error
1060 031_S_2018 error
1061 B11667035 error
1062 B55006637 error
1065 B83599353 error
1066 B25715232 error
1067 B52801561 error
1069 B48227553 error
1077 B97247975 error
1080 B42635533 error
1082 B82078560 error
1086 B30214599 error
1088 B73475423 error
1089 B87363937 error
1095 B26951105 error
1099 B43098693 error
1102 B63792019 error
1106 B24090258 error
1108 B81187857 error
1115 B90575129 error
1125 B65836649 error
1127 B98388895 error
1128 B65781664 error
1130 B17525647 error
1131 B26883230 error
1132 B61130259 error
1134 B82590204 error
1137 B13681621 error
1139 B44794062 error
1141 B46123892 error
1143 B32037155 error
1144 B29314228 error
1146 B26506745 error
1147 B99260348 error
1149 B78516379 error
1153 B34001495 error
1155 B98604009 error
1156 B42756185 error
1159 B32085845 error
1161 B64896348 error
1162 B47152746 error
1163 B39460060 error
1164 003_S_4441 error
1165 022_S_

2076 B69947521 error
2077 B38475589 error
2081 B94562023 error
2082 B74254198 error
2087 073_S_0089 error
2090 B64097756 error
2092 B97500167 error
2100 B62963368 error
2102 B92584337 error
2103 B11503021 error
2104 B98627375 error
2106 B58761418 error
2107 B54963910 error
2109 B94667740 error
2113 B70977963 error
2115 B96048145 error
2121 B61927724 error
2122 B32840128 error
2123 B83170117 error
2124 B95793592 error
2126 B39857656 error
2127 B33878424 error
2128 B91528620 error
2129 B20847910 error
2132 B46774482 error
2134 B85647756 error
2135 B45096540 error
2137 B54910005 error
2138 B94419915 error
2139 B66320183 error
2143 B21466161 error
2148 B84420603 error
2150 B13374109 error
2153 B10720460 error
2159 B63574447 error
2161 B39307056 error
2162 B96745125 error
2163 B32275111 error
2164 128_S_0135 error
2165 B94858957 error
2171 B85596885 error
2172 B36411607 error
2175 B61519626 error
2176 B76372157 error
2179 B98563970 error
2182 B54236260 error
2188 B86472788 error
2189 B54535

3053 B10474364 error
3056 B38574136 error
3058 B76342677 error
3061 B48223356 error
3065 B60121794 error
3066 B94953588 error
3071 B44197660 error
3072 B68586050 error
3075 B60115081 error
3078 B70268024 error
3086 B29700097 error
3090 B82229966 error
3097 B32336116 error
3099 B29850204 error
3100 B12612807 error
3101 B71045781 error
3105 B27410963 error
3106 B55191322 error
3107 B65880484 error
3109 B22537724 error
3112 B49085636 error
3115 B97890993 error
3118 B80114324 error
3119 B25997384 error
3120 B86838929 error
3122 B63308634 error
3123 127_S_0112 error
3125 B78548470 error
3126 003_S_2374 error
3128 B76617030 error
3129 B80463433 error
3133 B17190663 error
3141 B49403063 error
3142 B59594943 error
3144 B31055219 error
3145 B10197194 error
3159 B74328072 error
3164 B89474904 error
3166 B13536626 error
3167 B60324791 error
3168 B16069984 error
3169 B63857038 error
3177 B36874638 error
3178 B47309675 error
3179 B65428816 error
3181 B83597636 error
3182 B58336776 error
3183 B18420

4068 B10594635 error
4072 B38848592 error
4074 B65186366 error
4077 B85693841 error
4078 B69682621 error
4079 B52970639 error
4081 021_S_6978 error
4082 B98230141 error
4089 B28921939 error
4090 B14016331 error
4092 B25703979 error
4093 B98241639 error
4094 B63152407 error
4105 B40067953 error
4106 B63911664 error
4108 B71648555 error
4109 B12163141 error
4115 B80353193 error
4116 B11544718 error
4117 B84614270 error
4118 B89044340 error
4120 B81138926 error
4122 B96481536 error
4123 B47858685 error
4124 B82868543 error
4135 B44826377 error
4136 B17623548 error
4137 B82189482 error
4138 B82533431 error
4141 B34205680 error
4143 B17697191 error
4144 B41586550 error
4145 B42982161 error
4146 B66830066 error
4149 B53424484 error
4151 B92223444 error
4155 B65244102 error
4157 B50166298 error
4158 B95504281 error
4166 041_S_4004 error
4168 B86768925 error
4169 B26375382 error
4173 B62426845 error
4175 B32512144 error
4177 B63496946 error
4181 B10269249 error
4188 B96482424 error
4190 B61707

5095 B74858321 error
5099 B18371517 error
5100 B37464403 error
5101 B77123736 error
5106 B11963311 error
5108 053_S_4557 error
5109 B40298974 error
5110 B97240044 error
5111 B76202643 error
5112 B62102097 error
5113 B75082659 error
5115 B62326221 error
5120 B46595298 error
5122 B73283951 error
5124 B41880122 error
5131 B25986473 error
5132 B58842024 error
5133 B83473676 error
5136 B14057870 error
5139 B80202813 error
5140 B14331746 error
5141 B47511909 error
5147 B37105702 error
5154 B77262465 error
5155 B47317519 error
5157 B25726927 error
5163 B28164897 error
5165 B70299785 error
5168 B77211272 error
5172 B60721999 error
5175 B62565687 error
5176 B25603360 error
5180 B29038688 error
5181 B68102365 error
5182 B60701396 error
5183 B19496904 error
5188 B43571699 error
5191 B55104276 error
5192 129_S_4220 error
5193 B61235862 error
5200 B49731201 error
5201 B31548493 error
5202 B10149443 error
5206 B89108723 error
5207 B63580375 error
5215 B50610981 error
5219 B12806498 error
5220 B18983

6003 B78782605 error
6009 B37019413 error
6010 B36036790 error
6015 B47790811 error
6016 B50072721 error
6017 B76309364 error
6021 B62579289 error
6024 B48309631 error
6027 B24483454 error
6033 B17648113 error
6034 B35818998 error
6035 B55593099 error
6039 B36976155 error
6049 B64751382 error
6053 B67131157 error
6055 B98737188 error
6056 B25735560 error
6057 B77849321 error
6063 B90313419 error
6071 B27040996 error
6072 B55055889 error
6074 B85053420 error
6080 B60904865 error
6082 B36001244 error
6083 B73677676 error
6085 B21494740 error
6092 141_S_4426 error
6094 002_S_4225 error
6095 B20093346 error
6097 B92872041 error
6098 B82289687 error
6101 153_S_6679 error
6103 B30196143 error
6104 B15268431 error
6105 B99586844 error
6106 B82492086 error
6112 B71728664 error
6115 B11171441 error
6121 B36914973 error
6122 B85732666 error
6123 B51940871 error
6126 B35535582 error
6131 B23302827 error
6132 B38433155 error
6134 B45452649 error
6135 B89972311 error
6136 B25747608 error
6137 B6091

7021 B75594674 error
7022 B58902381 error
7025 B15773811 error
7031 B43342955 error
7036 B53006953 error
7038 B84753839 error
7043 B14442341 error
7044 B95334005 error
7047 B17522168 error
7050 B85301718 error
7053 B47149385 error
7054 B36482595 error
7057 B37315445 error
7061 B91484392 error
7063 B87384193 error
7066 B83944186 error
7071 B76135087 error
7073 B47561468 error
7074 B84665222 error
7075 B32966350 error
7077 B24361828 error
7079 B55797169 error
7080 B75018078 error
7081 B81179650 error
7082 B90077295 error
7084 B32658386 error
7091 B91654122 error
7092 B69529107 error
7093 B45863159 error
7095 082_S_4090 error
7098 B45052966 error
7099 018_S_4889 error
7102 B94823120 error
7104 B21161221 error
7107 B18346832 error
7108 B16252289 error
7115 B28268052 error
7117 B44477629 error
7126 B94548172 error
7128 B86175192 error
7130 B79634982 error
7133 B76737750 error
7135 B67104879 error
7136 B14313660 error
7138 B10093965 error
7139 B20610494 error
7141 B51911152 error
7153 B36179

In [21]:
mri_meta[mri_meta.Phase == 'ADNI GO'].Type.value_counts()

Processed    1192
Original      940
Name: Type, dtype: int64

In [22]:
pet_mri_tab.shape

(7490, 26)

In [23]:
pet_mri_tab = pet_mri_tab[pet_mri_tab['MRI_Age'].notna()]

In [24]:
pet_mri_tab = pet_mri_tab[pet_mri_tab['Age']>0]
pet_mri_tab.reset_index(drop=True, inplace = True)

In [25]:
pet_mri_tab = pet_mri_tab[pet_mri_tab['MRI_Type'] == 'Processed']
pet_mri_tab.reset_index(drop=True, inplace = True)

In [27]:
if (pet_mri_tab['Subject.ID'] == pet_mri_tab['MRI_subID']).all():
    print("✅ All Subject.ID values match MRI_subID.")
else:
    print("❌ There are mismatches between Subject.ID and MRI_subID.")

✅ All Subject.ID values match MRI_subID.


In [28]:
pet_mri_tab.shape

(3716, 26)

In [29]:
pet_mri_tab.Phase.value_counts()

ADNI 2     1807
ADNI GO     248
Name: Phase, dtype: int64

In [30]:
pet_mri_tab.Project.value_counts()

ADNI    2055
A4      1661
Name: Project, dtype: int64

In [31]:
pet_mri_tab

,Subject.ID,Project,Phase,Sex,Weight,Research.Group,VISCODE,Study.Date,Age,Modality,Description,Imaging.Protocol,Image.Data.ID,modality_subtype,PATH,MRI_subID,MRI_Modality,MRI_Research.Group,MRI_Age,MRI_VISCODE,MRI_studydate,MRI_Description,MRI_Imaging.protocol,MRI_Type,MRI_ID,MRI_PATH
0,B59563700,A4,NaN,M,83.9,LEARN amyloidNE,SCV2,1/01/2016,84.4,PET,Florbetapir <- MNI_3DBRAIN_C),NaN,I1323673,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,B59563700,MRI,LEARN amyloidNE,84.0,SCV4,1/01/2016,T1; GradWarp; DeFaced <- Sagittal 3D Accelerat...,NaN,Processed,I1384728,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
1,023_S_4115,ADNI,ADNI 2,M,84.9,LMCI,v21,9/30/2013,69.7,PET,"AV45 Co-registered, Averaged <- PET Brain AV_4...",SIEMENS,I393755,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,023_S_4115,MRI,LMCI,69.7,v21,9/30/2013,MT1; GradWarp; N3m <- MPRAGE,SIEMENS,Processed,I398283,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
2,032_S_4429,ADNI,ADNI 2,M,75.7,CN,v21,1/22/2014,79.2,PET,"AV45 Co-registered, Averaged <- ADNI Brain PET...",Siemens ECAT,I413047,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,032_S_4429,MRI,CN,79.2,v21,1/21/2014,MT1; GradWarp; N3m <- MPRAGE,SIEMENS,Processed,I416035,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
3,B45524671,A4,NaN,M,71.4,amyloidE,SCV2,1/01/2016,82.5,PET,Florbetapir <- PET Scan,NaN,I1411194,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,B45524671,MRI,amyloidE,82.6,SCV4,1/01/2016,T1; DeFaced <- Sagittal 3D Accelerated MPRAGE,NaN,Processed,I1364073,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
4,B14220479,A4,NaN,F,0.0,LEARN amyloidNE,SCV2,1/01/2016,71.8,PET,Florbetapir <- PET Scan,NaN,I1411241,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,B14220479,MRI,LEARN amyloidNE,71.0,SCV4,1/01/2016,T1; DeFaced <- Sagittal 3D Accelerated MPRAGE,NaN,Processed,I1384417,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3711,067_S_0056,ADNI,ADNI 2,F,76.7,CN,v31,1/07/2015,78.9,PET,"AV45 Co-registered, Averaged <- AV45_Dyn_4x5mi...",SIEMENS,I468830,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,067_S_0056,MRI,CN,78.8,v31,12/15/2014,MT1; GradWarp; N3m <- MPRAGE,SIEMENS,Processed,I468856,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
3712,005_S_0553,ADNI,ADNI GO,M,65.3,CN,m48,12/15/2010,89.3,PET,"AV45 Co-registered, Averaged <- 20 min 3D AV45...",Siemens/CTI,I209032,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,005_S_0553,MRI,CN,89.2,m48,10/13/2010,MT1; GradWarp; N3m <- MP-RAGE REPEAT,GE MEDICAL SYSTEMS,Processed,I322397,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
3713,006_S_4153,ADNI,ADNI 2,M,72.1,AD,v21,9/20/2013,81.5,PET,"AV45 Co-registered, Averaged <- Adni AV45 4 X ...",Siemens/CTI,I391868,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,006_S_4153,MRI,AD,81.5,v21,9/16/2013,MT1; N3m <- MPRAGE,Philips Medical Systems,Processed,I391062,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
3714,009_S_0751,ADNI,ADNI 2,M,57.5,CN,v31,9/17/2014,79.1,PET,"AV45 Co-registered, Averaged <- ADNIGO - AV45 ...",GE MEDICAL SYSTEMS,I445087,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,009_S_0751,MRI,CN,79.1,v31,9/04/2014,MT1; GradWarp; N3m <- MPRAGE,SIEMENS,Processed,I451346,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...


In [333]:
# add path to pet scans normalised to whole cerebellum, script cerebellum_normalisation. 
for i in range(0,np.shape(pet_mri_tab)[0]):
    name_pet = pet_mri_tab['Image.Data.ID'][i]+'_normalised.nii'
    filename = fnmatch.filter(os.listdir('/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25_10_23_registered_normalised_pet/'), name_pet)
    if len(filename)> 0:
        pet_mri_tab.loc[i,'PET_PATH_normalised'] = '/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25_10_23_registered_normalised_pet/' + filename[0]
    
    name_mri = pet_mri_tab['MRI_ID'][i]+'_registered.nii'
    filename = fnmatch.filter(os.listdir('/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25_10_23_registered_mri/'), name_mri)
    if len(filename)> 0:
        pet_mri_tab.loc[i,'MRI_PATH_registered'] = '/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25_10_23_registered_mri/' + filename[0]
     
    #print(i)

In [335]:
pet_mri_tab.shape

(4515, 27)

In [334]:
pet_mri_tab[pet_mri_tab['MRI_PATH_registered'].notna()]

,Subject.ID,Project,Phase,Sex,Weight,Research.Group,VISCODE,Study.Date,Age,Modality,Description,Imaging.Protocol,Image.Data.ID,modality_subtype,PATH,MRI_subID,MRI_Modality,MRI_Research.Group,MRI_Age,MRI_VISCODE,MRI_studydate,MRI_Description,MRI_Imaging.protocol,MRI_Type,MRI_ID,MRI_PATH,MRI_PATH_registered
0,B59563700,A4,NaN,M,83.9,LEARN amyloidNE,SCV2,1/01/2016,84.4,PET,Florbetapir <- MNI_3DBRAIN_C),NaN,I1323673,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,B59563700,MRI,LEARN amyloidNE,84.0,SCV4,1/01/2016,T1; GradWarp; DeFaced <- Sagittal 3D Accelerat...,NaN,Processed,I1384728,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
1,023_S_4115,ADNI,ADNI 2,M,84.9,LMCI,v21,9/30/2013,69.7,PET,"AV45 Co-registered, Averaged <- PET Brain AV_4...",SIEMENS,I393755,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,023_S_4115,MRI,LMCI,69.7,v21,9/30/2013,MT1; GradWarp; N3m <- MPRAGE,SIEMENS,Processed,I398283,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
3,032_S_4429,ADNI,ADNI 2,M,75.7,CN,v21,1/22/2014,79.2,PET,"AV45 Co-registered, Averaged <- ADNI Brain PET...",Siemens ECAT,I413047,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,032_S_4429,MRI,CN,79.2,v21,1/21/2014,MT1; GradWarp; N3m <- MPRAGE,SIEMENS,Processed,I416035,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
4,B45524671,A4,NaN,M,71.4,amyloidE,SCV2,1/01/2016,82.5,PET,Florbetapir <- PET Scan,NaN,I1411194,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,B45524671,MRI,amyloidE,82.6,SCV4,1/01/2016,T1; DeFaced <- Sagittal 3D Accelerated MPRAGE,NaN,Processed,I1364073,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
5,B14220479,A4,NaN,F,0.0,LEARN amyloidNE,SCV2,1/01/2016,71.8,PET,Florbetapir <- PET Scan,NaN,I1411241,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,B14220479,MRI,LEARN amyloidNE,71.0,SCV4,1/01/2016,T1; DeFaced <- Sagittal 3D Accelerated MPRAGE,NaN,Processed,I1384417,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4508,067_S_0056,ADNI,ADNI 2,F,76.7,CN,v31,1/07/2015,78.9,PET,"AV45 Co-registered, Averaged <- AV45_Dyn_4x5mi...",SIEMENS,I468830,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,067_S_0056,MRI,CN,78.8,v31,12/15/2014,MT1; GradWarp; N3m <- MPRAGE,SIEMENS,Processed,I468856,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
4509,005_S_0553,ADNI,ADNI GO,M,65.3,CN,m48,12/15/2010,89.3,PET,"AV45 Co-registered, Averaged <- 20 min 3D AV45...",Siemens/CTI,I209032,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,005_S_0553,MRI,CN,89.2,m48,10/13/2010,MT1; GradWarp; N3m <- MP-RAGE REPEAT,GE MEDICAL SYSTEMS,Processed,I322397,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
4510,006_S_4153,ADNI,ADNI 2,M,72.1,AD,v21,9/20/2013,81.5,PET,"AV45 Co-registered, Averaged <- Adni AV45 4 X ...",Siemens/CTI,I391868,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,006_S_4153,MRI,AD,81.5,v21,9/16/2013,MT1; N3m <- MPRAGE,Philips Medical Systems,Processed,I391062,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
4512,009_S_0751,ADNI,ADNI 2,M,57.5,CN,v31,9/17/2014,79.1,PET,"AV45 Co-registered, Averaged <- ADNIGO - AV45 ...",GE MEDICAL SYSTEMS,I445087,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,009_S_0751,MRI,CN,79.1,v31,9/04/2014,MT1; GradWarp; N3m <- MPRAGE,SIEMENS,Processed,I451346,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...


In [117]:
pet_mri_tab = pet_mri_tab[pet_mri_tab['PET_PATH_normalised'].notna()]
pet_mri_tab.reset_index(drop=True, inplace = True)
pet_mri_tab = pet_mri_tab[pet_mri_tab['MRI_PATH_registered'].notna()]
pet_mri_tab.reset_index(drop=True, inplace = True)

In [118]:
#pet_mri_tab = pd.read_csv('pet_mri_pairs.csv')

In [119]:
pet_mri_tab.shape

(3451, 28)

In [360]:
pet_mri_tab['MRI_ID'].value_counts()

I494162     2
I498008     2
I1363560    1
I1364044    1
I1384602    1
           ..
I424734     1
I362972     1
I1384522    1
I1364673    1
I291002     1
Name: MRI_ID, Length: 3714, dtype: int64

In [361]:
pet_mri_tab

,Subject.ID,Project,Phase,Sex,Weight,Research.Group,VISCODE,Study.Date,Age,Modality,Description,Imaging.Protocol,Image.Data.ID,modality_subtype,PATH,MRI_subID,MRI_Modality,MRI_Research.Group,MRI_Age,MRI_VISCODE,MRI_studydate,MRI_Description,MRI_Imaging.protocol,MRI_Type,MRI_ID,MRI_PATH
0,B59563700,A4,NaN,M,83.9,LEARN amyloidNE,SCV2,1/01/2016,84.4,PET,Florbetapir <- MNI_3DBRAIN_C),NaN,I1323673,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,B59563700,MRI,LEARN amyloidNE,84.0,SCV4,1/01/2016,T1; GradWarp; DeFaced <- Sagittal 3D Accelerat...,NaN,Processed,I1384728,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
1,023_S_4115,ADNI,ADNI 2,M,84.9,LMCI,v21,9/30/2013,69.7,PET,"AV45 Co-registered, Averaged <- PET Brain AV_4...",SIEMENS,I393755,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,023_S_4115,MRI,LMCI,69.7,v21,9/30/2013,MT1; GradWarp; N3m <- MPRAGE,SIEMENS,Processed,I398283,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
2,032_S_4429,ADNI,ADNI 2,M,75.7,CN,v21,1/22/2014,79.2,PET,"AV45 Co-registered, Averaged <- ADNI Brain PET...",Siemens ECAT,I413047,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,032_S_4429,MRI,CN,79.2,v21,1/21/2014,MT1; GradWarp; N3m <- MPRAGE,SIEMENS,Processed,I416035,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
3,B45524671,A4,NaN,M,71.4,amyloidE,SCV2,1/01/2016,82.5,PET,Florbetapir <- PET Scan,NaN,I1411194,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,B45524671,MRI,amyloidE,82.6,SCV4,1/01/2016,T1; DeFaced <- Sagittal 3D Accelerated MPRAGE,NaN,Processed,I1364073,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
4,B14220479,A4,NaN,F,0.0,LEARN amyloidNE,SCV2,1/01/2016,71.8,PET,Florbetapir <- PET Scan,NaN,I1411241,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,B14220479,MRI,LEARN amyloidNE,71.0,SCV4,1/01/2016,T1; DeFaced <- Sagittal 3D Accelerated MPRAGE,NaN,Processed,I1384417,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3711,067_S_0056,ADNI,ADNI 2,F,76.7,CN,v31,1/07/2015,78.9,PET,"AV45 Co-registered, Averaged <- AV45_Dyn_4x5mi...",SIEMENS,I468830,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,067_S_0056,MRI,CN,78.8,v31,12/15/2014,MT1; GradWarp; N3m <- MPRAGE,SIEMENS,Processed,I468856,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
3712,005_S_0553,ADNI,ADNI GO,M,65.3,CN,m48,12/15/2010,89.3,PET,"AV45 Co-registered, Averaged <- 20 min 3D AV45...",Siemens/CTI,I209032,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,005_S_0553,MRI,CN,89.2,m48,10/13/2010,MT1; GradWarp; N3m <- MP-RAGE REPEAT,GE MEDICAL SYSTEMS,Processed,I322397,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
3713,006_S_4153,ADNI,ADNI 2,M,72.1,AD,v21,9/20/2013,81.5,PET,"AV45 Co-registered, Averaged <- Adni AV45 4 X ...",Siemens/CTI,I391868,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,006_S_4153,MRI,AD,81.5,v21,9/16/2013,MT1; N3m <- MPRAGE,Philips Medical Systems,Processed,I391062,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
3714,009_S_0751,ADNI,ADNI 2,M,57.5,CN,v31,9/17/2014,79.1,PET,"AV45 Co-registered, Averaged <- ADNIGO - AV45 ...",GE MEDICAL SYSTEMS,I445087,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,009_S_0751,MRI,CN,79.1,v31,9/04/2014,MT1; GradWarp; N3m <- MPRAGE,SIEMENS,Processed,I451346,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...


In [300]:
meta4[meta4['Subject.ID']=='018_S_4597']

,Unnamed: 0,Unnamed: 0.1,Subject.ID,Project,Phase,Sex,Weight,Research.Group,Visit,Study.Date,Archive.Date,Age,Modality,Description,Type,Imaging.Protocol,Image.ID,VISCODE,Image.Data.ID,modality_subtype,PATH,year
27,30,44036,018_S_4597,ADNI,ADNI 2,F,55.5,EMCI,ADNI2 Year 1 Visit,3/11/2013,3/13/2013,69.5,MRI,MT1; N3m <- MPRAGE SENSE2,Processed,Philips Medical Systems,362978,v11,I362978,MPRAGE SENSE2,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,2013
448,649,44035,018_S_4597,ADNI,ADNI 2,F,55.5,EMCI,ADNI2 Year 1 Visit,3/11/2013,3/11/2013,69.5,MRI,MPRAGE SENSE2,Original,Acquisition Plane=SAGITTAL;Slice Thickness=1.2...,362532,v11,I362532,MPRAGE SENSE2,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,2013
2498,3612,44024,018_S_4597,ADNI,ADNI 2,F,56.7,EMCI,ADNI2 Month 3 MRI-New Pt,8/07/2012,8/07/2012,68.9,MRI,MPRAGE,Original,Acquisition Plane=SAGITTAL;Slice Thickness=1.2...,321921,v04,I321921,MPRAGE,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,2012
5666,8253,44010,018_S_4597,ADNI,ADNI 2,F,59.0,EMCI,ADNI2 Screening MRI-New Pt,3/14/2012,3/19/2012,68.5,MRI,MT1; N3m <- MPRAGE SENSE2,Processed,Philips Medical Systems,291022,v02,I291022,MPRAGE SENSE2,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,2012
8736,12727,44039,018_S_4597,ADNI,ADNI 2,F,55.5,EMCI,ADNI2 Year 1 Visit,3/11/2013,3/11/2013,69.5,MRI,MPRAGE,Original,Acquisition Plane=SAGITTAL;Slice Thickness=1.2...,362535,v11,I362535,MPRAGE,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,2013
10815,15796,44008,018_S_4597,ADNI,ADNI 2,F,59.0,EMCI,ADNI2 Screening MRI-New Pt,3/14/2012,4/25/2012,68.5,MRI,MT1; N3m <- MPRAGE,Processed,Philips Medical Systems,300508,v02,I300508,MPRAGE,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,2012
12548,18386,44029,018_S_4597,ADNI,ADNI 2,F,56.7,EMCI,ADNI2 Month 3 MRI-New Pt,8/07/2012,8/07/2012,68.9,MRI,MPRAGE SENSE2,Original,Acquisition Plane=SAGITTAL;Slice Thickness=1.2...,321926,v04,I321926,MPRAGE SENSE2,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,2012
19825,29042,44014,018_S_4597,ADNI,ADNI 2,F,57.6,EMCI,ADNI2 Baseline-New Pt,6/05/2012,6/19/2012,68.8,PET,"AV45 Co-registered, Averaged <- 3D BRAIN AV-45...",Processed,GEMS,311324,v03,I311324,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,2012
26671,39086,44009,018_S_4597,ADNI,ADNI 2,F,59.0,EMCI,ADNI2 Screening MRI-New Pt,3/14/2012,3/14/2012,68.5,MRI,MPRAGE SENSE2,Original,Acquisition Plane=SAGITTAL;Slice Thickness=1.2...,290307,v02,I290307,MPRAGE SENSE2,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,2012
27444,40159,44052,018_S_4597,ADNI,ADNI 2,F,51.7,EMCI,ADNI2 Year 2 Visit,4/22/2014,4/25/2014,70.6,PET,"AV45 Co-registered, Averaged <- 3D BRAIN AV-45...",Processed,GEMS,421678,v21,I421678,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,2014


In [324]:
pet_mri_tab[pet_mri_tab['MRI_ID'] == 'I498008']

,Subject.ID,Project,Phase,Sex,Weight,Research.Group,VISCODE,Study.Date,Age,Modality,Description,Imaging.Protocol,Image.Data.ID,modality_subtype,PATH,MRI_subID,MRI_Modality,MRI_Research.Group,MRI_Age,MRI_VISCODE,MRI_studydate,MRI_Description,MRI_Imaging.protocol,MRI_Type,MRI_ID,MRI_PATH
267,016_S_5031,ADNI,ADNI 2,F,66.7,EMCI,v21,5/14/2015,82.8,PET,"AV45 Co-registered, Averaged <- 016-S-5031 (AC...",SIEMENS,I500606,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,016_S_5031,MRI,EMCI,82.7,v21,5/04/2015,MT1; GradWarp; N3m <- Sag IR-SPGR,GE MEDICAL SYSTEMS,Processed,I498008,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
1728,016_S_5031,ADNI,ADNI 2,F,73.0,EMCI,v31,4/26/2016,83.7,PET,"AV45 Co-registered, Averaged <- 016-s-5031(AC)...",SIEMENS,I715555,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,016_S_5031,MRI,EMCI,82.7,v21,5/04/2015,MT1; GradWarp; N3m <- Sag IR-SPGR,GE MEDICAL SYSTEMS,Processed,I498008,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...


In [316]:
pet_mri_tab[pet_mri_tab['Subject.ID'] == '018_S_4597']

,Subject.ID,Project,Phase,Sex,Weight,Research.Group,VISCODE,Study.Date,Age,Modality,Description,Imaging.Protocol,Image.Data.ID,modality_subtype,PATH,MRI_subID,MRI_Modality,MRI_Research.Group,MRI_Age,MRI_VISCODE,MRI_studydate,MRI_Description,MRI_Imaging.protocol,MRI_Type,MRI_ID,MRI_PATH
3265,018_S_4597,ADNI,ADNI 2,F,57.6,EMCI,v03,6/05/2012,68.8,PET,"AV45 Co-registered, Averaged <- 3D BRAIN AV-45...",GEMS,I311324,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,018_S_4597,MRI,EMCI,68.9,v04,8/07/2012,MT1; N3m <- MPRAGE SENSE2,Philips Medical Systems,Processed,I334102,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
4571,018_S_4597,ADNI,ADNI 2,F,51.7,EMCI,v21,4/22/2014,70.6,PET,"AV45 Co-registered, Averaged <- 3D BRAIN AV-45...",GEMS,I421678,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,,,,NaN,,,,,,,


In [327]:
pet_mri_tab.VISCODE.value_counts()

SCV2    1661
v03      753
v21      503
bl       354
init     281
y2       244
v41      212
v11      156
v31      100
m48       82
v06       65
y4        44
m60       36
v51       18
m36        6
Name: VISCODE, dtype: int64

In [328]:
pet_mri_tab.MRI_VISCODE.value_counts()

SCV4    1661
v02      620
v21      493
init     275
y2       240
        ... 
y1        11
v05        9
m36        6
m06        1
y3         1
Name: MRI_VISCODE, Length: 22, dtype: int64

In [329]:
pet_mri_tab[pet_mri_tab.VISCODE == pet_mri_tab.MRI_VISCODE]

,Subject.ID,Project,Phase,Sex,Weight,Research.Group,VISCODE,Study.Date,Age,Modality,Description,Imaging.Protocol,Image.Data.ID,modality_subtype,PATH,MRI_subID,MRI_Modality,MRI_Research.Group,MRI_Age,MRI_VISCODE,MRI_studydate,MRI_Description,MRI_Imaging.protocol,MRI_Type,MRI_ID,MRI_PATH
1,023_S_4115,ADNI,ADNI 2,M,84.9,LMCI,v21,9/30/2013,69.7,PET,"AV45 Co-registered, Averaged <- PET Brain AV_4...",SIEMENS,I393755,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,023_S_4115,MRI,LMCI,69.7,v21,9/30/2013,MT1; GradWarp; N3m <- MPRAGE,SIEMENS,Processed,I398283,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
2,941_S_4036,ADNI,ADNI 3,M,88.0,EMCI,init,1/19/2018,80.7,PET,"AV45 Co-registered, Averaged <- ADNI3-BRAIN AV45",GE MEDICAL SYSTEMS,I966647,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,941_S_4036,MRI,EMCI,80.6,init,11/27/2017,Accelerated Sagittal MPRAGE,Acquisition Plane=SAGITTAL;Slice Thickness=1.2...,Original,I939342,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
3,032_S_4429,ADNI,ADNI 2,M,75.7,CN,v21,1/22/2014,79.2,PET,"AV45 Co-registered, Averaged <- ADNI Brain PET...",Siemens ECAT,I413047,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,032_S_4429,MRI,CN,79.2,v21,1/21/2014,MT1; GradWarp; N3m <- MPRAGE,SIEMENS,Processed,I416035,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
6,153_S_4372,ADNI,ADNI 2,F,100.5,CN,v21,12/04/2013,72.2,PET,"AV45 Co-registered, Averaged <- PET Brain_Flor...",SIEMENS,I401653,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,153_S_4372,MRI,CN,72.2,v21,12/11/2013,MT1; GradWarp; N3m <- MPRAGE,SIEMENS,Processed,I416106,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
7,016_S_4902,ADNI,ADNI 2,F,68.0,LMCI,v31,12/29/2015,78.8,PET,"AV45 Co-registered, Averaged <- 016-S-4902(AC)...",SIEMENS,I600580,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,016_S_4902,MRI,LMCI,78.7,v31,12/18/2015,MT1; GradWarp; N3m <- Accelerated SAG IR-SPGR,GE MEDICAL SYSTEMS,Processed,I848420,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4508,067_S_0056,ADNI,ADNI 2,F,76.7,CN,v31,1/07/2015,78.9,PET,"AV45 Co-registered, Averaged <- AV45_Dyn_4x5mi...",SIEMENS,I468830,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,067_S_0056,MRI,CN,78.8,v31,12/15/2014,MT1; GradWarp; N3m <- MPRAGE,SIEMENS,Processed,I468856,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
4509,005_S_0553,ADNI,ADNI GO,M,65.3,CN,m48,12/15/2010,89.3,PET,"AV45 Co-registered, Averaged <- 20 min 3D AV45...",Siemens/CTI,I209032,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,005_S_0553,MRI,CN,89.2,m48,10/13/2010,MT1; GradWarp; N3m <- MP-RAGE REPEAT,GE MEDICAL SYSTEMS,Processed,I322397,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
4510,006_S_4153,ADNI,ADNI 2,M,72.1,AD,v21,9/20/2013,81.5,PET,"AV45 Co-registered, Averaged <- Adni AV45 4 X ...",Siemens/CTI,I391868,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,006_S_4153,MRI,AD,81.5,v21,9/16/2013,MT1; N3m <- MPRAGE,Philips Medical Systems,Processed,I391062,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
4511,941_S_4036,ADNI,ADNI 3,M,88.0,EMCI,y2,3/06/2020,82.9,PET,"AV45 Co-registered, Averaged <- ADNI3-BRAIN AV45",GE MEDICAL SYSTEMS,I1302729,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,941_S_4036,MRI,EMCI,82.8,y2,2/26/2020,Accelerated Sagittal MPRAGE,Acquisition Plane=SAGITTAL;Slice Thickness=1.0...,Original,I1296964,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...


In [52]:
# remove scans with bad registration
meta_qc = pd.read_csv('/csc/epitkane/home/atagmazi/ADDL_pipeline/scripts/registration/metafile_ADDLpipeline_abeta_mri_27_05_2025_afterQC.csv',header=[0], index_col=[0])
pet_mri_tab = pet_mri_tab[pet_mri_tab['Image.Data.ID'].isin(meta_qc['Image.Data.ID'])].copy()
pet_mri_tab.reset_index(drop=True, inplace = True)

In [53]:
pet_mri_tab

,Subject.ID,Project,Phase,Sex,Weight,Research.Group,VISCODE,Study.Date,Age,Modality,Description,Imaging.Protocol,Image.Data.ID,modality_subtype,PATH,MRI_subID,MRI_Modality,MRI_Research.Group,MRI_Age,MRI_VISCODE,MRI_studydate,MRI_Description,MRI_Imaging.protocol,MRI_Type,MRI_ID,MRI_PATH
0,B59563700,A4,NaN,M,83.9,LEARN amyloidNE,SCV2,1/01/2016,84.4,PET,Florbetapir <- MNI_3DBRAIN_C),NaN,I1323673,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,B59563700,MRI,LEARN amyloidNE,84.0,SCV4,1/01/2016,T1; GradWarp; DeFaced <- Sagittal 3D Accelerat...,NaN,Processed,I1384728,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
1,023_S_4115,ADNI,ADNI 2,M,84.9,LMCI,v21,9/30/2013,69.7,PET,"AV45 Co-registered, Averaged <- PET Brain AV_4...",SIEMENS,I393755,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,023_S_4115,MRI,LMCI,69.7,v21,9/30/2013,MT1; GradWarp; N3m <- MPRAGE,SIEMENS,Processed,I398283,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
2,032_S_4429,ADNI,ADNI 2,M,75.7,CN,v21,1/22/2014,79.2,PET,"AV45 Co-registered, Averaged <- ADNI Brain PET...",Siemens ECAT,I413047,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,032_S_4429,MRI,CN,79.2,v21,1/21/2014,MT1; GradWarp; N3m <- MPRAGE,SIEMENS,Processed,I416035,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
3,B45524671,A4,NaN,M,71.4,amyloidE,SCV2,1/01/2016,82.5,PET,Florbetapir <- PET Scan,NaN,I1411194,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,B45524671,MRI,amyloidE,82.6,SCV4,1/01/2016,T1; DeFaced <- Sagittal 3D Accelerated MPRAGE,NaN,Processed,I1364073,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
4,B14220479,A4,NaN,F,0.0,LEARN amyloidNE,SCV2,1/01/2016,71.8,PET,Florbetapir <- PET Scan,NaN,I1411241,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,B14220479,MRI,LEARN amyloidNE,71.0,SCV4,1/01/2016,T1; DeFaced <- Sagittal 3D Accelerated MPRAGE,NaN,Processed,I1384417,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3421,067_S_0056,ADNI,ADNI 2,F,76.7,CN,v31,1/07/2015,78.9,PET,"AV45 Co-registered, Averaged <- AV45_Dyn_4x5mi...",SIEMENS,I468830,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,067_S_0056,MRI,CN,78.8,v31,12/15/2014,MT1; GradWarp; N3m <- MPRAGE,SIEMENS,Processed,I468856,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
3422,005_S_0553,ADNI,ADNI GO,M,65.3,CN,m48,12/15/2010,89.3,PET,"AV45 Co-registered, Averaged <- 20 min 3D AV45...",Siemens/CTI,I209032,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,005_S_0553,MRI,CN,89.2,m48,10/13/2010,MT1; GradWarp; N3m <- MP-RAGE REPEAT,GE MEDICAL SYSTEMS,Processed,I322397,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
3423,006_S_4153,ADNI,ADNI 2,M,72.1,AD,v21,9/20/2013,81.5,PET,"AV45 Co-registered, Averaged <- Adni AV45 4 X ...",Siemens/CTI,I391868,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,006_S_4153,MRI,AD,81.5,v21,9/16/2013,MT1; N3m <- MPRAGE,Philips Medical Systems,Processed,I391062,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
3424,009_S_0751,ADNI,ADNI 2,M,57.5,CN,v31,9/17/2014,79.1,PET,"AV45 Co-registered, Averaged <- ADNIGO - AV45 ...",GE MEDICAL SYSTEMS,I445087,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,009_S_0751,MRI,CN,79.1,v31,9/04/2014,MT1; GradWarp; N3m <- MPRAGE,SIEMENS,Processed,I451346,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...


In [ ]:
#split data on train and test subsets
train_size = 0.8
train_end = int(len(pet_mri_tab)*train_size)
t = int(0.8*train_end) #!!
v = int(0.2*train_end) 
#DATADIR = r"/csc/epitkane/data/ADNI/AD_DL_03_11_2021/"
print(f'train set size = {t}')

In [ ]:
'''pet_mri_tab['pet_min'] = 0
pet_mri_tab['pet_max'] = 0
pet_mri_tab['pet_95q'] = 0
pet_mri_tab['pet_99q'] = 0

pet_mri_tab['mri_min'] = 0
pet_mri_tab['mri_max'] = 0
pet_mri_tab['mri_95q'] = 0
pet_mri_tab['mri_99q'] = 0


i_invalid = []
pet_all_values = []
mri_all_values = []
for i in range(pet_mri_tab.shape[0]):
    img_pet = nib.load(pet_mri_tab['PET_PATH_normalised'][i])
        
    img_mri = nib.load(pet_mri_tab['MRI_PATH_registered'][i])
    
    image_pet = np.asarray(img_pet.get_fdata())
    image_mri = np.asarray(img_mri.get_fdata())
    
    if np.isnan(image_pet).any() or np.isnan(image_mri).any():
        print(f"Skipping index {i}: NaNs detected in PET or MRI scan.")
        i_invalid.append(i)
    elif np.max(image_pet) <= 0 or np.max(image_mri) <= 0:  # Changed from == 0 to <= 0
        print(f"Skipping index {i}: Empty or non-positive PET or MRI scan.")
        i_invalid.append(i)
    else:
        pet_mri_tab.loc[i,'pet_min'] = np.min(image_pet)
        pet_mri_tab.loc[i,'pet_max'] = np.max(image_pet)
        pet_mri_tab.loc[i,'pet_99q'] = np.percentile(image_pet, 99)
        pet_mri_tab.loc[i,'pet_999q'] = np.percentile(image_pet, 99.9)
        
        # Store min & max values (ensures outliers are captured)
        #pet_all_values.append(image_pet.min())
        while i<=t:
            pet_all_values.append(image_pet.max())
        # Sample a subset of pixels (reduces memory usage)
            pet_all_values.extend(np.random.choice(image_pet.flatten(), size=min(10000, image_pet.size), replace=False))
        #pet_all_values.extend(image_pet[image_pet > 0].flatten())


        
        pet_mri_tab.loc[i,'mri_min'] = np.min(image_mri)
        pet_mri_tab.loc[i,'mri_max'] = np.max(image_mri)
        pet_mri_tab.loc[i,'mri_99q'] = np.percentile(image_mri, 99)
        pet_mri_tab.loc[i,'mri_999q'] = np.percentile(image_mri, 99.9)
        # Store min & max values (ensures outliers are captured)
        #mri_all_values.append(image_mri.min())
        while i <=t:
            mri_all_values.append(image_mri.max())
        # Sample a subset of pixels (reduces memory usage)
            mri_all_values.extend(np.random.choice(image_mri.flatten(), size=min(10000, image_mri.size), replace=False))
        #mri_all_values.extend(image_mri[image_mri > 0].flatten())

        
    print(i)
    
print(i_invalid)    
pet_minimum = np.min(pet_mri_tab['pet_min'])
pet_maximum = np.max(pet_mri_tab['pet_max'])
mri_minimum = np.min(pet_mri_tab['mri_min'])
mri_maximum = np.max(pet_mri_tab['mri_max'])
print(f'MinMax computation is done! PET minimum is {pet_minimum}, PET maximum is {pet_maximum}. MRI minimum is {mri_minimum}, MRI maximum is {mri_maximum}')
'''

/tmp/ipykernel_43436/1921735778.py:27: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2.908689022064209' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  pet_mri_tab.loc[i,'pet_max'] = np.max(image_pet)
/tmp/ipykernel_43436/1921735778.py:28: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1.7029719293117513' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  pet_mri_tab.loc[i,'pet_95q'] = np.percentile(image_pet, 95)
/tmp/ipykernel_43436/1921735778.py:29: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2.2386636161804194' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  pet_mri_tab.loc[i,'pet_99q'] = np.percentile(image_pet, 99)
/

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52


/tmp/ipykernel_43436/1921735778.py:31: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1.8371753692626953' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  pet_mri_tab.loc[i,'mri_min'] = np.min(image_mri)


53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
1861
1862
1863
1864
1865
1866
1867
1868
1869
1870
1871
1872
1873
1874
1875
1876
1877
1878
1879
1880
1881
1882
1883
1884
1885
1886
1887
1888
1889
1890
1891
1892
1893
1894
1895
1896
1897
1898
1899
1900
1901
1902
1903
1904
1905
1906
1907
1908
1909
1910
1911
1912
1913
1914
1915
1916
1917
1918
1919
1920
1921
1922
1923
1924
1925
1926
1927
1928
1929
1930
1931
1932
1933
1934
1935
1936
1937
1938
1939
1940
1941
1942
1943
1944
1945
1946
1947
1948
1949
1950
1951
1952
1953
1954
1955
1956
1957
1958
1959
1960
1961
1962
1963
1964
1965
1966
1967
1968
1969
1970
1971
1972
1973
1974
1975
1976
1977
1978
1979
1980
1981
1982
1983
1984
1985
1986
1987
1988
1989
1990
1991
1992
1993
1994
1995
1996
1997
1998
1999
2000
2001
2002
2003
2004
2005
2006
2007
2008
200

In [ ]:

logging.basicConfig(level=logging.INFO, format='%(message)s')
logger = logging.getLogger(__name__)
def process_row(i, pet_path, mri_path):
    result = {
        'i': i,
        'pet_min': 0,
        'pet_max': 0,
        'pet_90q': 0,
        'pet_95q': 0,
        'pet_99q': 0,
        'pet_999q': 0,
        'mri_min': 0,
        'mri_max': 0,
        'mri_90q': 0,
        'mri_95q': 0,
        'mri_99q': 0,
        'mri_999q': 0,
        'pet_values': [],
        'mri_values': [],
        'invalid': False
    }
    logger.info(f"Processing index {i}")  # Log each index being processed

    try:
        image_pet = np.asarray(nib.load(pet_path).get_fdata())
        image_mri = np.asarray(nib.load(mri_path).get_fdata())
        
        if np.isnan(image_pet).any() or np.isnan(image_mri).any():
            result['invalid'] = True
            return result
        elif np.max(image_pet) <= 0 or np.max(image_mri) <= 0:
            result['invalid'] = True
            return result
        
        result['pet_min'] = np.min(image_pet)
        result['pet_max'] = np.max(image_pet)
        result['pet_90q'] = np.percentile(image_pet, 90)
        result['pet_95q'] = np.percentile(image_pet, 95)
        result['pet_99q'] = np.percentile(image_pet, 99)
        result['pet_999q'] = np.percentile(image_pet, 99.9)
        if i <= t:
            result['pet_values'] = np.random.choice(image_pet.flatten(), size=min(10000, image_pet.size), replace=False)
        
        result['mri_min'] = np.min(image_mri)
        result['mri_max'] = np.max(image_mri)
        result['mri_90q'] = np.percentile(image_mri, 90)
        result['mri_95q'] = np.percentile(image_mri, 95)
        result['mri_99q'] = np.percentile(image_mri, 99)
        result['mri_999q'] = np.percentile(image_mri, 99.9)
        if i <= t:
            result['mri_values'] = np.random.choice(image_mri.flatten(), size=min(10000, image_mri.size), replace=False)
        
    except Exception as e:
        print(f"Error processing index {i}: {e}")
        result['invalid'] = True
    
    return result

# Number of workers to run in parallel (adjust based on your CPU)
n_jobs = 8

results = Parallel(n_jobs=n_jobs)(
    delayed(process_row)(i, pet_mri_tab['PET_PATH_normalised'][i], pet_mri_tab['MRI_PATH_registered'][i])
    for i in range(pet_mri_tab.shape[0])
)

# Rebuild dataframe from results
for res in results:
    i = res['i']
    if not res['invalid']:
        pet_mri_tab.loc[i, 'pet_min'] = res['pet_min']
        pet_mri_tab.loc[i, 'pet_max'] = res['pet_max']
        pet_mri_tab.loc[i, 'pet_90q'] = res['pet_90q']
        pet_mri_tab.loc[i, 'pet_95q'] = res['pet_95q']
        pet_mri_tab.loc[i, 'pet_99q'] = res['pet_99q']
        pet_mri_tab.loc[i, 'pet_999q'] = res['pet_999q']
        
        pet_mri_tab.loc[i, 'mri_min'] = res['mri_min']
        pet_mri_tab.loc[i, 'mri_max'] = res['mri_max']
        pet_mri_tab.loc[i, 'mri_90q'] = res['mri_90q']
        pet_mri_tab.loc[i, 'mri_95q'] = res['mri_95q']
        pet_mri_tab.loc[i, 'mri_99q'] = res['mri_99q']
        pet_mri_tab.loc[i, 'mri_999q'] = res['mri_999q']
        
pet_all_values = np.concatenate([r['pet_values'] for r in results if not r['invalid']])
mri_all_values = np.concatenate([r['mri_values'] for r in results if not r['invalid']])
i_invalid = [r['i'] for r in results if r['invalid']]

# Final min/max stats
print(f"Invalid indexes: {i_invalid}")
print(f"PET min: {pet_mri_tab['pet_min'].min()}, max: {pet_mri_tab['pet_max'].max()}")
print(f"MRI min: {pet_mri_tab['mri_min'].min()}, max: {pet_mri_tab['mri_max'].max()}")


In [1]:
# Drop invalid rows
pet_mri_tab.drop(index=i_invalid, inplace=True)
pet_mri_tab.reset_index(drop=True, inplace=True)

print(f"Remaining valid rows: {pet_mri_tab.shape[0]}")

NameError: name 'pet_mri_tab' is not defined

In [ ]:
pet_all_values = np.array(pet_all_values)
mri_all_values = np.array(mri_all_values)

p_quant90 = np.quantile(pet_all_values, 0.90)
m_quant90 = np.quantile(mri_all_values, 0.90)

p_quant95 = np.quantile(pet_all_values, 0.95)
m_quant95 = np.quantile(mri_all_values, 0.95)


p_quant99 = np.quantile(pet_all_values, 0.99)
m_quant99 = np.quantile(mri_all_values, 0.99)

p_quant999 = np.quantile(pet_all_values, 0.999)
m_quant999 = np.quantile(mri_all_values, 0.999)

p_mean = np.mean(pet_all_values)
m_mean = np.mean(mri_all_values)

p_std = np.std(pet_all_values)
m_std = np.std(mri_all_values)

p_median = np.median(pet_all_values)
m_median = np.median(mri_all_values)

p_iqr = iqr(pet_all_values, rng=(25, 75))
m_iqr = iqr(mri_all_values, rng=(25, 75))

p_after_clip = np.clip(pet_all_values, 0, p_quant999, out=None)
m_after_clip = np.clip(mri_all_values, 0, m_quant999, out=None)
p_mean_clip = np.mean(p_after_clip)
m_mean_clip = np.mean(m_after_clip)
p_std_clip = np.std(p_after_clip)
m_std_clip = np.std(m_after_clip)
p_min_clip = np.min(p_after_clip)
m_min_clip = np.min(m_after_clip)
p_max_clip = np.max(p_after_clip)
m_max_clip = np.max(m_after_clip)


np.savez("stats_train.npz",
         p_quant90=p_quant90,
         m_quant90=m_quant90,
         p_quant95=p_quant95,
         m_quant95=m_quant95,
         p_quant99=p_quant99, 
         m_quant99=m_quant99,
         p_quant999=p_quant999, 
         m_quant999=m_quant999,
         p_mean=p_mean, 
         m_mean=m_mean, 
         p_std=p_std, 
         m_std=m_std,
         p_median=p_median, 
         m_median=m_median, 
         p_iqr=p_iqr, 
         m_iqr=m_iqr,
         p_mean_clip=p_mean_clip, 
         m_mean_clip=m_mean_clip,
         p_std_clip=p_std_clip, 
         m_std_clip=m_std_clip,
         p_mim_clip=p_min_clip, 
         m_min_clip=m_min_clip,
         p_max_clip=p_max_clip, 
         m_max_clip=m_max_clip
        )


In [ ]:
plt.figure(figsize=(10, 5))

# PET Histogram
plt.subplot(2, 2, 1)
plt.hist(pet_all_values, bins=500, color='blue', alpha=0.7, edgecolor='black')
plt.xlabel("Pixel Value")
plt.ylabel("Frequency")
plt.title("PET Image Histogram")

# MRI Histogram
plt.subplot(2, 2, 2)
plt.hist(mri_all_values, bins=500, color='red', alpha=0.7, edgecolor='black')
plt.xlabel("Pixel Value")
plt.ylabel("Frequency")
plt.title("MRI Image Histogram")


plt.subplot(2, 2, 3)
plt.hist(np.log1p(pet_all_values), bins=500, color='blue', alpha=0.7, edgecolor='black')  # Log scale on Y-axis
plt.xlabel("Log Pixel Value")
plt.ylabel("Frequency")
plt.title("PET Image Histogram")



plt.subplot(2, 2, 4)
plt.hist(np.log1p(mri_all_values), bins=500, color='red', alpha=0.7, edgecolor='black')
plt.xlabel("Log Pixel Value")
plt.ylabel("Frequency")
plt.title("MRI Image Histogram")


# Save and show the plot
plt.tight_layout()
plt.savefig("pixels_value_histograms.png")  # Save the figure
plt.show()  # Display the plot

In [ ]:
plt.figure(figsize=(10, 5))

# PET Histogram
plt.subplot(2, 2, 1)
plt.hist(p_after_clip, bins=500, color='blue', alpha=0.7, edgecolor='black')
plt.xlabel("Pixel Value")
plt.ylabel("Frequency")
plt.title("PET Image Histogram")

# MRI Histogram
plt.subplot(2, 2, 2)
plt.hist(m_after_clip, bins=500, color='red', alpha=0.7, edgecolor='black')
plt.xlabel("Pixel Value")
plt.ylabel("Frequency")
plt.title("MRI Image Histogram")


plt.subplot(2, 2, 3)
plt.hist(np.log1p(p_after_clip), bins=500, color='blue', alpha=0.7, edgecolor='black')  # Log scale on Y-axis
plt.xlabel("Log Pixel Value")
plt.ylabel("Frequency")
plt.title("PET Image Histogram")



plt.subplot(2, 2, 4)
plt.hist(np.log1p(m_after_clip), bins=500, color='red', alpha=0.7, edgecolor='black')
plt.xlabel("Log Pixel Value")
plt.ylabel("Frequency")
plt.title("MRI Image Histogram")


# Save and show the plot
plt.tight_layout()
plt.savefig("pixels_value_histograms_clip999.png")  # Save the figure
plt.show()  # Display the plot

In [22]:
i_invalid #pet_mri_tab.drop(index = i_invalid)

[]

In [41]:
#pet_mri_tab = pet_mri_tab[pet_mri_tab.mri_min >= 0]
pet_mri_tab.reset_index(drop=True, inplace = True)

In [42]:
pet_minimum = np.min(pet_mri_tab['pet_min'])
pet_maximum = np.max(pet_mri_tab['pet_max'])
mri_minimum = np.min(pet_mri_tab['mri_min'])
mri_maximum = np.max(pet_mri_tab['mri_max'])
print(f'MinMax computation is done! PET minimum is {pet_minimum}, PET maximum is {pet_maximum}. MRI minimum is {mri_minimum}, MRI maximum is {mri_maximum}')


MinMax computation is done! PET minimum is 0.0, PET maximum is 1270.4830322265625. MRI minimum is 0.0, MRI maximum is 26299.69921875


In [32]:
pet_mri_tab.to_csv('pet_mri_pairs.csv')

In [44]:
pet_mri_tab.shape

(3471, 30)

In [45]:
suc

4983

In [46]:
er

3232

In [47]:
pet_meta[pet_meta['Subject.ID'] == 'B10256269']

,Unnamed: 0,Subject.ID,Project,Phase,Sex,Weight,Research.Group,Visit,Study.Date,Archive.Date,Age,Modality,Description,Type,Imaging.Protocol,Image.ID,VISCODE,Image.Data.ID,modality_subtype,PATH,year
3742,217585,B10256269,A4,NaN,F,60.3,LEARN amyloidNE,SCV2,1/01/2016,7/20/2020,65.8,PET,Florbetapir <- LZAZ- (AC),Processed,NaN,1321180,SCV2,I1321180,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,2016


In [48]:
pet_mri_tab[pet_mri_tab['Subject.ID'] == 'B10256269']

,Subject.ID,Project,Phase,Sex,Weight,Research.Group,VISCODE,Age,Modality,Description,Imaging.Protocol,Image.Data.ID,modality_subtype,PATH,MRI_subID,MRI_Modality,MRI_Research.Group,MRI_Age,MRI_VISCODE,MRI_Description,MRI_Imaging.protocol,MRI_Type,MRI_ID,MRI_PATH,MRI_PATH_registered,PET_PATH_normalised,pet_min,pet_max,mri_min,mri_max
1990,B10256269,A4,NaN,F,60.3,LEARN amyloidNE,SCV2,65.8,PET,Florbetapir <- LZAZ- (AC),NaN,I1321180,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,B10256269,MRI,LEARN amyloidNE,66.0,SCV4,T1; GradWarp; DeFaced <- Sagittal 3D Accelerat...,NaN,Processed,I1384839,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,0.0,2.555523,0.206524,471.969879


In [49]:
pet_mri_tab.Phase.value_counts()

ADNI 2     1749
ADNI GO     210
ADNI 1       24
Name: Phase, dtype: int64

In [50]:
pet_mri_tab.Project.value_counts()

ADNI    1983
A4      1488
Name: Project, dtype: int64

In [22]:
pet_mri_tab[pet_mri_tab.Phase == 'ADNI 1'].MRI_Type.value_counts()

Processed    132
Name: MRI_Type, dtype: int64

In [23]:
pet_mri_tab[pet_mri_tab.Phase == 'ADNI 2'].MRI_Type.value_counts()

Processed    1840
Name: MRI_Type, dtype: int64

In [24]:
pet_mri_tab[pet_mri_tab.Phase == 'ADNI GO'].MRI_Type.value_counts()

Processed    251
Name: MRI_Type, dtype: int64

In [25]:
pet_mri_tab[pet_mri_tab.Phase == 'ADNI 3'].MRI_Type.value_counts()

Original    971
Name: MRI_Type, dtype: int64

In [26]:
pet_mri_tab[pet_mri_tab.Project == 'A4'].MRI_Type.value_counts()

Processed    1788
Name: MRI_Type, dtype: int64

In [165]:
pet_mri_tab.MRI_Modality.value_counts()

MRI    4982
Name: MRI_Modality, dtype: int64

In [ ]:
pet_mri_tab[pet_mri_tab.Phase == 'ADNI 1'][pet_mri_tab.MRI_Type == 'Processed']

In [31]:
pet_mri_tab[pet_mri_tab.Phase == 'ADNI 3'][pet_mri_tab.MRI_Type == 'Processed']

/csc/epitkane/home/atagmazi/.conda/envs/atagmazi_gpu8/lib/python3.7/site-packages/ipykernel_launcher.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  """Entry point for launching an IPython kernel.


,Subject.ID,Project,Phase,Sex,Weight,Research.Group,VISCODE,Age,Modality,Description,Imaging.Protocol,Image.Data.ID,modality_subtype,PATH,MRI_subID,MRI_Modality,MRI_Research.Group,MRI_Age,MRI_VISCODE,MRI_Description,MRI_Imaging.protocol,MRI_Type,MRI_ID,MRI_PATH


In [32]:
pet_mri_tab[pet_mri_tab['Research.Group'] == 'amyloidE']

,Subject.ID,Project,Phase,Sex,Weight,Research.Group,VISCODE,Age,Modality,Description,Imaging.Protocol,Image.Data.ID,modality_subtype,PATH,MRI_subID,MRI_Modality,MRI_Research.Group,MRI_Age,MRI_VISCODE,MRI_Description,MRI_Imaging.protocol,MRI_Type,MRI_ID,MRI_PATH
3731,B10081264,A4,NaN,F,72.6,amyloidE,SCV2,67.9,PET,Florbetapir <- PET Scan,NaN,I1410362,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,B10081264,MRI,amyloidE,67.9,SCV4,T1; GradWarp; DeFaced <- Sagittal 3D Accelerat...,NaN,Processed,I1364445,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
3733,B10102783,A4,NaN,F,0.0,amyloidE,SCV2,72.6,PET,Florbetapir <- PET Scan,NaN,I1410368,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,B10102783,MRI,amyloidE,72.7,SCV4,T1; DeFaced <- Sagittal 3D Accelerated MPRAGE,NaN,Processed,I1363140,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
3735,B10108368,A4,NaN,F,69.0,amyloidE,SCV2,66.8,PET,Florbetapir <- PET Scan,NaN,I1410363,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,B10108368,MRI,amyloidE,66.9,SCV4,T1; GradWarp; DeFaced <- Sagittal 3D Accelerat...,NaN,Processed,I1364240,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
3736,B10117310,A4,NaN,M,112.1,amyloidE,SCV2,69.4,PET,Florbetapir <- FRAME 1,NaN,I1324825,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,B10117310,MRI,amyloidE,69.5,SCV4,T1; GradWarp; DeFaced <- Sagittal 3D Accelerat...,NaN,Processed,I1363742,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
3748,B10350512,A4,NaN,M,0.0,amyloidE,SCV2,70.9,PET,Florbetapir <- PET Scan,NaN,I1410364,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,B10350512,MRI,amyloidE,71.1,SCV4,T1; GradWarp; DeFaced <- Sagittal 3D Accelerat...,NaN,Processed,I1364363,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8196,B99696480,A4,NaN,F,95.3,amyloidE,SCV2,74.6,PET,Florbetapir <- LZAZ-34 (AC),NaN,I1324700,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,B99696480,MRI,amyloidE,74.6,SCV4,T1; GradWarp; DeFaced <- Sagittal 3D Accelerat...,NaN,Processed,I1364178,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
8199,B99711873,A4,NaN,M,62.1,amyloidE,SCV2,73.4,PET,Florbetapir <- PET Scan,NaN,I1410706,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,B99711873,MRI,amyloidE,73.5,SCV4,T1; GradWarp; DeFaced <- Sagittal 3D Accelerat...,NaN,Processed,I1363372,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
8208,B99860801,A4,NaN,M,53.8,amyloidE,SCV2,71.7,PET,"Florbetapir <- [AC],,,2,,,",NaN,I1324682,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,B99860801,MRI,amyloidE,71.8,SCV4,T1; DeFaced <- Accelerated Sag IR-FSPGR,NaN,Processed,I1363279,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...
8209,B99870985,A4,NaN,M,0.0,amyloidE,SCV2,71.4,PET,Florbetapir <- PET Scan,NaN,I1411052,AV45,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...,B99870985,MRI,amyloidE,71.4,SCV4,T1; DeFaced <- Sagittal 3D Accelerated MPRAGE,NaN,Processed,I1362921,/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25...


In [33]:
pet_mri_tab[pet_mri_tab.Project == 'ADNI'].MRI_Description.value_counts()

Accelerated Sagittal MPRAGE             662
MT1; GradWarp; N3m <- MPRAGE            444
MT1; GradWarp; N3m <- MPRAGE GRAPPA2    400
MT1; N3m <- MPRAGE                      202
MT1; N3m <- MPRAGE SENSE2               180
                                       ... 
MT1; N3m <- MPRAGE REPEAT ASO             1
MT1; N3m <- ADNI SH    MPRAGE ASOX2       1
MPR; ; N3 <- ADNI SH    MPRAGE ASO        1
MT1; GradWarp; N3m <- MP-RAGE repeat      1
3D MPRAGE                                 1
Name: MRI_Description, Length: 84, dtype: int64

In [34]:
import seaborn as sns
import matplotlib.pyplot as plt

In [35]:
pet_mri_tab['Research.Group'].value_counts().values

array([1247, 1150,  722,  541,  487,  364,  251,  220])

In [36]:
pet_mri_tab['Research.Group'].value_counts()

amyloidE           1247
CN                 1150
EMCI                722
LEARN amyloidNE     541
MCI                 487
LMCI                364
AD                  251
SMC                 220
Name: Research.Group, dtype: int64

In [ ]:
plt.figure(figsize=(15,6))

ax = sns.violinplot(x="Research.Group", y="Age", data=pet_mri_tab)

# Calculate number of obs per group & median to position labels
medians = pet_mri_tab.groupby(['Research.Group'])['Age'].max().values
nobs = pet_mri_tab['Research.Group'].value_counts(sort=False).values
nobs = [str(x) for x in nobs.tolist()]
nobs = ["n: " + i for i in nobs]
 
# Add text to the figure
pos = range(len(nobs))
for tick, label in zip(pos, ax.get_xticklabels()):
   ax.text(pos[tick], medians[tick] + 0.03, nobs[tick],
            horizontalalignment='left',
            size=13,
            color='blue',
            weight='semibold')
#plt.show()
plt.savefig('samples_dist.png')

In [ ]:
pd.set_option('display.max_rows', None)
pet_mri_tab[pet_mri_tab.Project == 'A4'].Age.value_counts()

In [ ]:
pet_mri_tab[pet_mri_tab.Age == 0.0]

In [ ]:
pet_mri_tab.Age 